# 1. Design a spiral object

A `Spiral` object is a class that can be used to progressively build spiral-shaped objects amenable to DNA origami designs. It starts empty, and different segments from pre-made templates can be progressively added, with desired characteristics such as lengths or radii. The following notebook shows how this works.

In [1]:
from autorigami.spiral_generation import Spiral, load_curves
from autorigami.viz import show_curves, show_spiral

## Straight segments

The first segment needs a length and two radii. One turn is 2.65 nm high, which keeps the turns a small distance above the 2.6 nm limit.

In [2]:
spiral = Spiral()
stem = spiral.add_straight_segment(40.0, x_radius=16.0, y_radius=16.0, rise_per_turn=2.65)
show_spiral(spiral)

The `validate` applies a validation algorithm against both of our constraints: the turning angle between consecutive edges (3.25° maximum) and the distance between two parts of the helix (2.6 nm minimum).

In [3]:
print(spiral.validate())

valid
curve 0: 0 curvature violations (largest angle 1.22 deg), 0 separation violations


## Y segments

A Y segment divides a round tube into two arms. It has a middle part, where the tube becomes wide, and a top part, where the two arms become round. Each part has its own opening angle: the angle between the vertical axis and the axis of an arm.

The segment starts at the terminus of the stem.

In [4]:
y = stem.termini[0].add_y_segment(
    middle_length=40.0,
    top_length=40.0,
    middle_opening_angle_deg=38.66,
    top_opening_angle_deg=23.71,
)
show_spiral(spiral, highlight=y)

## Segments on the arms

A Y segment has two termini, one for each arm. A straight segment on a terminus keeps the radii of that terminus.

In [5]:
arm = y.termini[0].add_straight_segment(25.0)
show_spiral(spiral, highlight=arm)

## Funnel segments

A funnel segment changes the radii linearly, from the radii of the terminus to the end radii that you give.

In [6]:
funnel = y.termini[1].add_funnel_segment(25.0, end_x_radius=10.0, end_y_radius=10.0)
show_spiral(spiral, highlight=funnel)

The `spiral.segments` method lists all segments, so `spiral.segments[1].termini[0]` is the same terminus as `y.termini[0]`.

In [7]:
[type(segment).__name__ for segment in spiral.segments]

['StraightSegment', 'YSegment', 'StraightSegment', 'FunnelSegment']

## Curves

One DNA helix cannot follow two arms. Thus the segment on `y.termini[1]` starts a second curve. The unique curves that compose a design can be visualized with the `spiral.curves` method.

In [8]:
curves = spiral.curves()
show_curves(curves)

## Validate

`validate` on a `spiral` object with multiple curves now examines each curve, and also the distance between each pair of curves.

In [9]:
result = spiral.validate()
print(result)

NOT VALID
curve 0: 120 curvature violations (largest angle 26.01 deg), 159801 separation violations (smallest distance 0.220 nm)
curve 1: 0 curvature violations (largest angle 1.99 deg), 1141 separation violations (smallest distance 2.560 nm)
curves 0 and 1: 52 separation violations (smallest distance 2.505 nm)


This spiral is not valid, and that is the expected result at this stage. The middle part of a Y segment has sharp corners at its waist, and its turns are nearer than 2.6 nm. Gravity packing will repair these faults: see the next notebook.

## Save

`save` writes the curves to one file in numpy format, and `load_curves` reads them again.

In [10]:
spiral.save("../outputs/spiral_1.npz")
[curve.shape for curve in load_curves("../outputs/spiral_1.npz")]

[(21630, 3), (2314, 3)]